# Task 3: MLP Shape Reconstruction

This notebook is a Python version of the Task 3 PDF. It builds ellipse masks, trains an MLP to paint shapes onto a pixel grid, and includes separate answer sections for the challenge questions.

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

# Keep the notebook reproducible.
np.random.seed(42)
torch.manual_seed(42)


## 2. Problem Setup

In [ ]:
# Task 3 uses ellipse parameters as inputs and a binary image as output.
# Each training example is:
# [a, b, theta, cx, cy] -> flattened mask

img_size = 32  # Final image resolution used for the pixel painter demo
num_samples = 1000  # Enough examples to show the mapping without making the demo too heavy
x = np.linspace(-1.0, 1.0, img_size)
X, Y = np.meshgrid(x, x)

# We use a fixed grid so the network learns the same output layout every time.
input_dim = 5
output_dim = img_size * img_size

## 3. Ellipse Mask Helper

In [ ]:
def ellipse_mask(a, b, theta, cx, cy, X, Y):
    # Shift the grid so the ellipse center becomes the origin.
    x_shift = X - cx
    y_shift = Y - cy

    # Rotate coordinates into the ellipse's local frame.
    x_rot = x_shift * np.cos(theta) + y_shift * np.sin(theta)
    y_rot = -x_shift * np.sin(theta) + y_shift * np.cos(theta)

    # Points inside the ellipse satisfy the standard quadratic form.
    return ((x_rot / a) ** 2 + (y_rot / b) ** 2) <= 1.0

## 4. Generate Training Data

In [ ]:
# Parameter ranges follow the Task 3 prompt.
a = 0.2 + 0.4 * np.random.rand(num_samples)
b = 0.1 + 0.3 * np.random.rand(num_samples)
theta = np.pi * np.random.rand(num_samples)
cx = -0.5 + np.random.rand(num_samples)
cy = -0.5 + np.random.rand(num_samples)

# Stack the inputs in the exact order the network sees them.
X_train = np.column_stack([a, b, theta, cx, cy]).astype(np.float32)

# Normalize the input features so optimization is more stable.
X_mean = X_train.mean(axis=0)
X_std = X_train.std(axis=0) + 1e-8
X_train_norm = (X_train - X_mean) / X_std

# Build the target masks by rasterizing each ellipse onto the grid.
Y_train = np.zeros((num_samples, output_dim), dtype=np.float32)
for i in range(num_samples):
    mask = ellipse_mask(a[i], b[i], theta[i], cx[i], cy[i], X, Y)
    Y_train[i, :] = mask.astype(np.float32).ravel()

print("Training data shape:", X_train_norm.shape, Y_train.shape)

## 5. Build the MLP

In [ ]:
# This network follows the Task 3 idea: parameters in, pixels out.
# The final logits are passed through a sigmoid only when we want probabilities.
def build_model(l2_strength=0.0):
    model = nn.Sequential(
        nn.Linear(input_dim, 128),
        nn.ReLU(),
        nn.Linear(128, 256),
        nn.ReLU(),
        nn.Linear(256, output_dim),
    )
    model.weight_decay = l2_strength
    return model

def count_params(model):
    return sum(param.numel() for param in model.parameters())

model = build_model()
print(model)
print('Parameter count:', count_params(model))


## 6. Train the Network

In [ ]:
# We keep the demo short, but the structure matches the MATLAB prompt.
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=getattr(model, 'weight_decay', 0.0))
criterion = nn.BCEWithLogitsLoss()
val_count = int(0.2 * len(X_train_norm))
train_count = len(X_train_norm) - val_count
X_fit = torch.tensor(X_train_norm[:train_count], dtype=torch.float32)
Y_fit = torch.tensor(Y_train[:train_count], dtype=torch.float32)
X_holdout = torch.tensor(X_train_norm[train_count:], dtype=torch.float32)
Y_holdout = torch.tensor(Y_train[train_count:], dtype=torch.float32)
fit_loader = DataLoader(TensorDataset(X_fit, Y_fit), batch_size=32, shuffle=True)

history = {'loss': [], 'val_loss': []}
for _ in range(30):
    model.train()
    epoch_losses = []
    for batch_X, batch_Y in fit_loader:
        optimizer.zero_grad()
        loss = criterion(model(batch_X), batch_Y)
        loss.backward()
        optimizer.step()
        epoch_losses.append(float(loss.item()))
    history['loss'].append(float(np.mean(epoch_losses)))
    model.eval()
    with torch.no_grad():
        val_loss = criterion(model(X_holdout), Y_holdout).item()
    history['val_loss'].append(float(val_loss))

print('Final training loss:', history['loss'][-1])
print('Final validation loss:', history['val_loss'][-1])


## 7. Test Cases

In [ ]:
# These test ellipses mirror the examples from the Task 3 PDF.
test_ellipses = np.array([
    [0.5, 0.3, np.pi / 4, 0.2, 0.2],
    [0.4, 0.2, 0.0, -0.3, -0.2],
    [0.6, 0.15, np.pi / 3, 0.3, -0.1],
    [0.35, 0.35, np.pi / 6, -0.2, 0.3],
], dtype=np.float32)

test_norm = (test_ellipses - X_mean) / X_std
with torch.no_grad():
    logits = model(torch.tensor(test_norm, dtype=torch.float32)).cpu().numpy()
predictions = 1.0 / (1.0 + np.exp(-logits))

# Generate ground truth masks for direct comparison.
ground_truth = []
for params in test_ellipses:
    mask = ellipse_mask(params[0], params[1], params[2], params[3], params[4], X, Y)
    ground_truth.append(mask.astype(np.float32))
ground_truth = np.array(ground_truth)


## 8. Visualize Predictions

In [ ]:
fig, axes = plt.subplots(3, len(test_ellipses), figsize=(4 * len(test_ellipses), 10))

for i, params in enumerate(test_ellipses):
    pred_mask = predictions[i].reshape(img_size, img_size)
    true_mask = ground_truth[i]
    diff_mask = np.abs(true_mask - pred_mask)

    axes[0, i].imshow(true_mask, cmap='gray', origin='lower', extent=[-1, 1, -1, 1])
    axes[0, i].set_title(f'True {i+1}')
    axes[0, i].set_axis_off()

    axes[1, i].imshow(pred_mask, cmap='gray', origin='lower', extent=[-1, 1, -1, 1])
    axes[1, i].set_title('Predicted')
    axes[1, i].set_axis_off()

    axes[2, i].imshow(diff_mask, cmap='magma', origin='lower', extent=[-1, 1, -1, 1])
    axes[2, i].set_title('Abs Diff')
    axes[2, i].set_axis_off()

    axes[0, i].text(0.02, 0.02, f'a={params[0]:.2f}\nb={params[1]:.2f}\nθ={params[2]:.2f}',
                    transform=axes[0, i].transAxes, fontsize=8, color='yellow',
                    bbox=dict(facecolor='black', alpha=0.5, pad=2))

plt.tight_layout()
plt.show()

## 9. Optional Regularization Experiment

In [ ]:
# This mirrors the optional Task 3 regularization question.
baseline_model = build_model(l2_strength=0.0)
regularized_model = build_model(l2_strength=0.01)

def fit_mask_model(model_to_fit):
    optimizer = torch.optim.Adam(model_to_fit.parameters(), lr=1e-3, weight_decay=getattr(model_to_fit, 'weight_decay', 0.0))
    criterion = nn.BCEWithLogitsLoss()
    loader = DataLoader(TensorDataset(X_fit, Y_fit), batch_size=32, shuffle=True)
    history_local = {'loss': [], 'val_loss': []}
    for _ in range(30):
        model_to_fit.train()
        losses = []
        for batch_X, batch_Y in loader:
            optimizer.zero_grad()
            loss = criterion(model_to_fit(batch_X), batch_Y)
            loss.backward()
            optimizer.step()
            losses.append(float(loss.item()))
        history_local['loss'].append(float(np.mean(losses)))
        model_to_fit.eval()
        with torch.no_grad():
            history_local['val_loss'].append(float(criterion(model_to_fit(X_holdout), Y_holdout).item()))
    return history_local

baseline_history = fit_mask_model(baseline_model)
regularized_history = fit_mask_model(regularized_model)

print('Baseline val loss:', baseline_history['val_loss'][-1])
print('Regularized val loss:', regularized_history['val_loss'][-1])


## 10. Answers to Task 3 Questions

### Q1: How does the training time compare to the circle example? What causes any differences?
Training is usually slower for ellipses because the input space is larger and more varied. The network now has to learn rotation, aspect ratio, and translation together, so the mapping is harder than the circle case.

### Q2: Are there any ellipse configurations that the network consistently fails to generate correctly? What might be the cause?
The hardest cases are usually thin ellipses, highly rotated ellipses, and ellipses near the image boundary. These are harder because small pixel changes matter more and the binary mask can be sensitive to discretization at 32 x 32 resolution.

### Q3: If you increase the image resolution from 16 x 16 to 32 x 32, what changes are needed?
The output layer must grow from 256 to 1024 neurons, the training targets must be flattened to length 1024, and the model may need more data or slightly larger hidden layers to keep the reconstruction quality stable.

### Optional regularization answer
L2 regularization usually helps when the model starts fitting noise or pixel-level artifacts too closely. A small value such as 0.001 or 0.01 is often a better starting point than a large value like 0.1 because too much regularization can underfit the shape details.